# A10c IndoBERT vs GOLD — Inference-Only Evaluation (GPU)

Mengevaluasi model **A7 IndoBERT** (aspek + polarity) yang sudah dilatih pada
silver terhadap label **human-gold** (`gold.jsonl`) — **tanpa training ulang dan
tanpa re-tune**: memakai kalibrasi beku (temperature + detection thresholds dari
silver validation) dan menerapkannya ke gold test split.

Ini referensi human-gold terpisah, BUKAN membuka ulang locked silver test.
Melengkapi `evaluate-gold-baselines` (keyword/TF-IDF) sehingga ketiga model
dibandingkan pada gold test yang sama.

Prasyarat: notebook `06` (A7 model) + `07` (kalibrasi) sudah ada di Drive,
dan `gold.jsonl` sudah di `SIPATURE/data/annotations/gold/`.


## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Step 2 — Konfigurasi path & parameter

In [ ]:
# ============================================================
# CONFIGURATION CELL — satu-satunya tempat mengubah parameter.
# ============================================================
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/SIPATURE")
DRIVE_SPLIT_DIR = DRIVE_ROOT / "data" / "splits"
DRIVE_GOLD_DIR = DRIVE_ROOT / "data" / "annotations" / "gold"

# A7 run (hasil notebook 06) + kalibrasi beku (hasil notebook 07).
MODEL_RUN_ID = "20260813-1050_indobert-silver-v1"
CALIBRATION_ID = f"{MODEL_RUN_ID}_calibration-v1"
GOLD_EVAL_ID = f"{MODEL_RUN_ID}_gold-v1"

MODEL_RUN_DIR_DRIVE = DRIVE_ROOT / "runs" / MODEL_RUN_ID
CALIBRATION_DIR_DRIVE = DRIVE_ROOT / "calibration" / CALIBRATION_ID

PROJECT_DIR = Path("/content/hackathon/ml")
GOLD_DIR = PROJECT_DIR / "data" / "annotations" / "gold"
SPLIT_DIR = PROJECT_DIR / "data" / "splits"
MODEL_RUN_DIR = PROJECT_DIR / "runs" / MODEL_RUN_ID
CALIBRATION_DIR = PROJECT_DIR / "calibration" / CALIBRATION_ID
OUTPUT_DIR = PROJECT_DIR / "artifacts" / GOLD_EVAL_ID

DRIVE_OUTPUT_DIR = DRIVE_ROOT / "evaluation" / GOLD_EVAL_ID
DRIVE_METRICS_DIR = DRIVE_ROOT / "metrics"

GOLD_FILE = "gold.jsonl"
SPLIT_FILES = [
    "train_silver_v1.jsonl",
    "validation_silver_v1.jsonl",
    "test_silver_v1.jsonl",
    "split_manifest_silver_v1.json",
]

print("Model run (Drive)  :", MODEL_RUN_DIR_DRIVE)
print("Calibration (Drive):", CALIBRATION_DIR_DRIVE)
print("Gold (Drive)       :", DRIVE_GOLD_DIR / GOLD_FILE)
print("Output (lokal)     :", OUTPUT_DIR)
print("Output (Drive)     :", DRIVE_OUTPUT_DIR)


## Step 3 — Clone repository dari GitHub

In [ ]:
from google.colab import userdata
import base64
import os
import shutil
import subprocess

token = userdata.get("GITHUB_TOKEN")
assert token, "GITHUB_TOKEN tidak ditemukan di Colab Secrets"

credentials = f"x-access-token:{token}"
authorization = base64.b64encode(credentials.encode()).decode()

repo_dir = "/content/hackathon"
shutil.rmtree(repo_dir, ignore_errors=True)

environment = os.environ.copy()
environment["GIT_CONFIG_COUNT"] = "1"
environment["GIT_CONFIG_KEY_0"] = "http.extraHeader"
environment["GIT_CONFIG_VALUE_0"] = f"Authorization: Basic {authorization}"

result = subprocess.run(
    ["git", "clone", "https://github.com/jodypangaribuan/hackathon.git", repo_dir],
    env=environment,
    text=True,
    capture_output=True,
)

print("Return code:", result.returncode)
print(result.stdout)
print(result.stderr)

assert result.returncode == 0, "Clone gagal. Periksa izin token GitHub."


## Step 4 — Verifikasi commit terbaru (git log)

In [ ]:
%cd /content/hackathon/ml
!git log --oneline -3


## Step 5 — Install dependencies

In [ ]:
%cd /content/hackathon/ml
!python -m pip uninstall -y torchvision
!python -m pip install -r requirements-colab.lock.txt
!python -m pip install --no-deps -e .


## Step 6 — Verifikasi GPU & versi package

In [ ]:
import importlib.util
import torch
import transformers

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA tersedia:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Torchvision ditemukan:", importlib.util.find_spec("torchvision") is not None)

from transformers import BertForSequenceClassification
print("BertForSequenceClassification berhasil diimpor: OK")


## Step 7 — Copy gold + split + model + kalibrasi dari Drive

In [ ]:
import shutil
from pathlib import Path

# gold
GOLD_DIR.mkdir(parents=True, exist_ok=True)
gold_source = DRIVE_GOLD_DIR / GOLD_FILE
assert gold_source.is_file(), f"gold.jsonl tidak ditemukan di Drive: {gold_source}"
shutil.copy2(gold_source, GOLD_DIR / GOLD_FILE)
print("Disalin:", GOLD_FILE)

# split
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
for filename in SPLIT_FILES:
    source = DRIVE_SPLIT_DIR / filename
    assert source.is_file(), f"Split file tidak ditemukan: {source}"
    shutil.copy2(source, SPLIT_DIR / filename)
    print("Disalin:", filename)

# model run (A7) — seluruh folder
MODEL_RUN_DIR.parent.mkdir(parents=True, exist_ok=True)
if MODEL_RUN_DIR.exists():
    shutil.rmtree(MODEL_RUN_DIR)
shutil.copytree(MODEL_RUN_DIR_DRIVE, MODEL_RUN_DIR)
print("Disalin model run:", MODEL_RUN_DIR.name)

# kalibrasi
CALIBRATION_DIR.parent.mkdir(parents=True, exist_ok=True)
if CALIBRATION_DIR.exists():
    shutil.rmtree(CALIBRATION_DIR)
shutil.copytree(CALIBRATION_DIR_DRIVE, CALIBRATION_DIR)
print("Disalin kalibrasi:", CALIBRATION_DIR.name)


## Step 8 — Import modul sipature_ml

In [ ]:
import sys
from pathlib import Path

source_dir = PROJECT_DIR / "src"
assert source_dir.is_dir(), "Folder source SIPATURE tidak ditemukan."
if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))

import sipature_ml
print("Modul SIPATURE berhasil dimuat dari:")
print(sipature_ml.__file__)


## Step 9 — Jalankan evaluasi IndoBERT vs gold

In [ ]:
import torch

from sipature_ml.indobert_gold import run_gold_indobert_evaluation

assert torch.cuda.is_available(), "CUDA GPU tidak tersedia."
assert not OUTPUT_DIR.exists(), f"Output dir sudah ada: {OUTPUT_DIR}"

metrics = run_gold_indobert_evaluation(
    SPLIT_DIR,
    MODEL_RUN_DIR,
    CALIBRATION_DIR,
    GOLD_DIR / GOLD_FILE,
    OUTPUT_DIR,
)

print("IndoBERT (gold) aspect Macro F1:", round(metrics["macro_f1"], 4))
print("IndoBERT (gold) aspect Micro F1:", round(metrics["micro_f1"], 4))
print("IndoBERT (gold) polarity Macro F1:", round(metrics["polarity"]["macro_f1"], 4))
print("Basis:", metrics["evaluation_basis"])


## Step 10 — Tampilkan per-aspect & perbandingan silver

In [ ]:
print("Per-aspect F1 (gold test):")
for aspect, info in sorted(metrics["per_aspect"].items()):
    print(f"  {aspect:<22} f1={info['f1']:.4f}  (support {info['support']})")

print("\nPerbandingan aspek (Macro F1):")
print("  IndoBERT silver (locked test) : 0.5247")
print(f"  IndoBERT gold   (inference)   : {metrics['macro_f1']:.4f}")


## Step 11 — Copy output ke Drive

In [ ]:
import shutil
from pathlib import Path

DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for source in sorted(OUTPUT_DIR.glob("*")):
    if source.is_file():
        shutil.copy2(source, DRIVE_OUTPUT_DIR / source.name)
        print(f"Disalin: {source.name} -> {DRIVE_OUTPUT_DIR}")

# Salin juga ke metrics/ agar bisa dibaca notebook perbandingan.
DRIVE_METRICS_DIR.mkdir(parents=True, exist_ok=True)
metric_file = OUTPUT_DIR / "indobert-gold-v1-test-metrics.json"
if metric_file.is_file():
    shutil.copy2(metric_file, DRIVE_METRICS_DIR / metric_file.name)
    print(f"Disalin: {metric_file.name} -> {DRIVE_METRICS_DIR}")


## Step 12 — Run summary (hash & metric)

In [ ]:
# ============================================================
# RUN SUMMARY — hash, metric, dan temuan.
# ============================================================
from pathlib import Path
from sipature_ml.manifest import sha256_file

print("REFERENCE LABEL TYPE:", metrics["reference_label_type"])
print("GOLD SHA256         :", metrics["gold_sha256"])
print("A7 RUN ID           :", metrics["a7_run_id"])
print("Aspect Macro F1     :", round(metrics["macro_f1"], 4))
print("Aspect Micro F1     :", round(metrics["micro_f1"], 4))
print("Polarity Macro F1   :", round(metrics["polarity"]["macro_f1"], 4))
print("Severity            :", metrics["severity"]["status"])

print("\nOUTPUT DIR (lokal):", OUTPUT_DIR)
print("OUTPUT DIR (Drive) :", DRIVE_OUTPUT_DIR)
print("\nTEMUAN: IndoBERT silver 0.5247 vs gold (angka di atas).")
print("Ini inference-only (model A7 dilatih silver, tanpa re-tune).")
